In [1]:
import os, sys, json
import numpy as np
import torch
import gymnasium as gym

print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.cuda.is_available()} | device = {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Gymnasium  : {gym.__version__}")

env = gym.make("BipedalWalker-v3")
obs, _ = env.reset()
print(f"\nObservation space : {env.observation_space}  →  shape {obs.shape}")
print(f"Action space      : {env.action_space}")
env.close()

Python    : 3.12.0 (main, Apr 25 2026, 21:54:59) [GCC 15.2.1 20260209]
PyTorch   : 2.12.0+cu130
CUDA      : True  |  device=NVIDIA GeForce RTX 3070
Gymnasium : 1.2.3

Observation space : Box([-3.1415927 -5.        -5.        -5.        -3.1415927 -5.
 -3.1415927 -5.        -0.        -3.1415927 -5.        -3.1415927
 -5.        -0.        -1.        -1.        -1.        -1.
 -1.        -1.        -1.        -1.        -1.        -1.       ], [3.1415927 5.        5.        5.        3.1415927 5.        3.1415927
 5.        5.        3.1415927 5.        3.1415927 5.        5.
 1.        1.        1.        1.        1.        1.        1.
 1.        1.        1.       ], (24,), float32)  →  shape (24,)
Action space      : Box(-1.0, 1.0, (4,), float32)


/home/mitchellds/studia/sem6/IOwADC/.venv2/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
from config import HYPERPARAMS, ARCHITECTURES, TOTAL_TIMESTEPS, N_SEEDS, DEVICE

print(f"Device          : {DEVICE}")
print(f"Timesteps/run   : {TOTAL_TIMESTEPS:,}")
print(f"Seeds/config    : {N_SEEDS}")
print(f"Total runs      : {len(HYPERPARAMS) * len(ARCHITECTURES) * N_SEEDS}")

print(f"\n── Hyperparameter sets ({len(HYPERPARAMS)}) " + "─"*40)
for name, hp in HYPERPARAMS.items():
    print(f"  {name}")
    for k, v in hp.items():
        print(f"    {k:<20} = {v}")

print(f"\n── Architectures ({len(ARCHITECTURES)}) " + "─"*44)
for name, arch in ARCHITECTURES.items():
    print(f"  {name}:  net_arch={arch['net_arch']},  activation={arch['activation_fn'].__name__}")

Device          : cuda
Total timesteps : 600,000 per run
Seeds per config: 10

Hyperparameter sets (3):
  hp_default: {'learning_rate': 0.0003, 'batch_size': 256, 'tau': 0.005, 'gamma': 0.99, 'learning_starts': 10000, 'buffer_size': 300000, 'ent_coef': 'auto'}
  hp_high_lr: {'learning_rate': 0.001, 'batch_size': 256, 'tau': 0.005, 'gamma': 0.99, 'learning_starts': 10000, 'buffer_size': 300000, 'ent_coef': 'auto'}
  hp_large_batch: {'learning_rate': 0.0003, 'batch_size': 512, 'tau': 0.02, 'gamma': 0.995, 'learning_starts': 10000, 'buffer_size': 300000, 'ent_coef': 'auto'}

Architectures (2):
  arch_small: net_arch=[64, 64], activation=ReLU
  arch_large: net_arch=[256, 256], activation=ReLU

Total runs : 60


In [ ]:
from train import run_benchmark
run_benchmark()

In [ ]:
from train import train_all
train_all(total_timesteps=TOTAL_TIMESTEPS, resume=False)

In [ ]:
from utils import load_results
import numpy as np

print(f"{'Config':<40} {'Seeds done':>10} {'Final mean±std':>20}")
print("-" * 72)

for hp_name in HYPERPARAMS:
    for arch_name in ARCHITECTURES:
        data = load_results("results", hp_name, arch_name)
        if not data:
            print(f"  {hp_name}__{arch_name:<30} {'—':>10}")
            continue
        final_mean = data["mean"][-1]
        final_std  = data["std"][-1]
        n = data["n_seeds"]
        print(f"  {hp_name}__{arch_name:<30} {n:>10}   {final_mean:+.2f} ± {final_std:.2f}")


In [ ]:
from plot import plot_hyperparams_comparison, plot_arch_comparison, plot_eval_overlay

plot_hyperparams_comparison(show=False)
plot_arch_comparison(show=False)

# Eval overlay requires evaluate.py --best to have been run first (see cell 7)
# plot_eval_overlay(show=False)

# Display inline (Jupyter only):
from IPython.display import Image, display
import os

for fname in ["hyperparams_comparison.png", "arch_comparison.png"]:
    path = os.path.join("plots", fname)
    if os.path.exists(path):
        display(Image(path))

In [ ]:
from evaluate import find_best_model, evaluate_single

# Evaluate all configs, find best, save best_config.json
best = find_best_model(n_episodes=20)

# %% [markdown]
# ### Eval overlay plot

# %%
from plot import plot_eval_overlay
plot_eval_overlay(show=False)

from IPython.display import Image, display
path = "plots/eval_overlay.png"
if os.path.exists(path):
    display(Image(path))

In [ ]:
import json

best_meta_path = "models/best_config.json"
if os.path.exists(best_meta_path):
    with open(best_meta_path) as f:
        meta = json.load(f)
    config = meta["config"]
    hp_name, arch_name = config.split("__", 1)
    # Pick the seed with highest mean reward
    best_seed = max(
        meta["per_seed"],
        key=lambda x: x["mean_reward"]
    )["seed"]
    print(f"Rendering: {hp_name} / {arch_name} / seed {best_seed}")
    # evaluate_single(hp_name, arch_name, best_seed, n_episodes=3, render=True)
else:
    print("Run cell 7 first.")